# LeetCode 4038 — Count Integers Appearing in a Single Block bằng Python

Link đề: https://leetcode.com/problems/count-integers-appearing-in-a-single-block/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Counting, Run-length

Skeleton LeetCode:

```python
class Solution:
    def countSpecialIntegers(self, nums: List[int]) -> int:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: số nguyên `x` là **đặc biệt** nếu **mọi lần xuất hiện** của `x` trong `nums` tạo thành **một khối liên tiếp duy nhất**; đếm số giá trị `x` **khác nhau** đặc biệt.
2. Biết khái niệm **khối liên tiếp (run)** và hai cách nhận biết: `lần_cuối - lần_đầu + 1 == số_lần`, hoặc đếm số khối của từng giá trị bằng 1.
3. Biết dùng `dict`/`Counter`, `itertools.groupby`, `enumerate` và quét một lượt với tập "đã đóng" (closed set).
4. Viết 4 cách (vị trí đầu/cuối + đếm, đếm số khối theo `groupby`, quét một lượt, brute force chuỗi con) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: đếm số khối thay vì số giá trị khác nhau, nhầm với "xuất hiện một lần", chỉ so với hàng xóm trái, dùng `set` làm mất thông tin vị trí.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Một số nguyên `x` gọi là **đặc biệt** nếu **tất cả các lần xuất hiện của `x` trong `nums` tạo thành một khối liên tiếp (một đoạn con liên tiếp duy nhất)**. Trả về **số lượng số nguyên khác nhau** là đặc biệt.

| Ví dụ | `nums` | Output | Giải thích |
|---|---|---|---|
| 1 | `[1,2,2,1]` | `1` | `1` xuất hiện ở chỉ số `0` và `3` (hai khối rời nhau) -> không đặc biệt; `2` chiếm một khối `[1,2]` -> đặc biệt. Đáp án `1` |
| 2 | `[3,3,1,2,2,1]` | `2` | `3` ở `[0,1]` (một khối) -> đặc biệt; `1` ở chỉ số `2` và `5` (hai khối) -> không; `2` ở `[3,4]` -> đặc biệt. Đáp án `2` |

Constraints: `1 <= len(nums) <= 100`; `1 <= nums[i] <= 100`.

Lưu ý: giá trị chỉ xuất hiện **một lần** cũng là một khối (độ dài 1) nên **đặc biệt**.

In [1]:
for nums in ([1, 2, 2, 1], [3, 3, 1, 2, 2, 1]):
    pos = {}
    for i, x in enumerate(nums):
        pos.setdefault(x, []).append(i)
    print(nums, pos)

[1, 2, 2, 1] {1: [0, 3], 2: [1, 2]}
[3, 3, 1, 2, 2, 1] {3: [0, 1], 1: [2, 5], 2: [3, 4]}


## 3. Kiến thức nền

### 3.1. Khối liên tiếp (run) của một giá trị

Một **khối** là đoạn con liên tiếp gồm các phần tử bằng nhau mà không thể mở rộng thêm. `[3,3,1,2,2,1]` có các khối: `3,3` | `1` | `2,2` | `1`. Giá trị `1` có **hai** khối nên không đặc biệt; `3` và `2` mỗi giá trị một khối.

In [2]:
from itertools import groupby
print([(v, len(list(g))) for v, g in groupby([3, 3, 1, 2, 2, 1])])

[(3, 2), (1, 1), (2, 2), (1, 1)]


### 3.2. `dict` lưu vị trí đầu và cuối

Duyệt `enumerate(nums)`; với mỗi giá trị `x`, lưu `first[x]` (lần đầu thấy, bằng `setdefault`), cập nhật `last[x] = i` mỗi lần, và đếm `cnt[x]`. Cho một giá trị: đặc biệt khi `last - first + 1 == cnt` (các vị trí lấp kín đoạn `[first, last]`).

In [3]:
nums = [3, 3, 1, 2, 2, 1]
first, last, cnt = {}, {}, {}
for i, x in enumerate(nums):
    first.setdefault(x, i)
    last[x] = i
    cnt[x] = cnt.get(x, 0) + 1
print(first, last, cnt)
print({x: last[x] - first[x] + 1 == cnt[x] for x in cnt})

{3: 0, 1: 2, 2: 3} {3: 1, 1: 5, 2: 4} {3: 2, 1: 2, 2: 2}
{3: True, 1: False, 2: True}


### 3.3. `collections.Counter`

`Counter(nums)` đếm số lần xuất hiện của từng giá trị: `Counter([1,2,2,1]) = {1: 2, 2: 2}`. Chỉ dùng được cho đếm; muốn kiểm tra **liên tiếp** cần thêm thông tin vị trí.

In [4]:
from collections import Counter
print(Counter([3, 3, 1, 2, 2, 1]))

Counter({3: 2, 1: 2, 2: 2})


### 3.4. `itertools.groupby`: đếm số khối mỗi giá trị

`groupby(nums)` gom các phần tử bằng nhau **liền kề**. Duyệt từng nhóm, tăng `blocks[v] += 1`. Giá trị có `blocks[v] == 1` là đặc biệt. Số giá trị khác nhau có `blocks == 1` chính là đáp án.

In [5]:
blocks = {}
for v, g in groupby([3, 3, 1, 2, 2, 1]):
    blocks[v] = blocks.get(v, 0) + 1
print(blocks, sum(1 for b in blocks.values() if b == 1))

{3: 1, 1: 2, 2: 1} 2


### 3.5. Tập đã đóng (`closed`): quét một lượt

Quét từ trái sang phải; khi một giá trị `prev` kết thúc khối (phần tử kế tiếp khác `prev`), ta **đóng** `prev` (thêm vào `closed`). Nếu sau này gặp lại một giá trị đã đóng thì nó **không đặc biệt** (đã có khối thứ hai). Cách này không cần từ điển vị trí.

In [6]:
closed, bad = set(), set()
prev = None
for x in [3, 3, 1, 2, 2, 1]:
    if x != prev:
        if prev is not None:
            closed.add(prev)
        if x in closed:
            bad.add(x)
    prev = x
print(closed, bad)

{1, 2, 3} {1}


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Với mỗi giá trị `x`, ta cần biết: **các vị trí của `x` có lấp kín một đoạn liên tiếp không**? Nếu các vị trí là `p_1 < p_2 < ... < p_c` thì chúng tạo một khối iff `p_c - p_1 + 1 = c` (đoạn `[p_1, p_c]` có đúng `c` ô, mà đã có `c` ô thuộc `x`, vậy không còn chỗ trống cho giá trị khác).

Tính tay `[3,3,1,2,2,1]`:

| giá trị | vị trí | `last - first + 1` | số lần | đặc biệt? |
|---|---|---|---|---|
| 3 | 0, 1 | 2 | 2 | có |
| 1 | 2, 5 | 4 | 2 | không |
| 2 | 3, 4 | 2 | 2 | có |

Đáp án `2`. Tính tay `[1,2,2,1]`: `1` ở `0, 3` -> `4 != 2` không; `2` ở `1, 2` -> `2 == 2` có -> `1`. Tính tay `[5]`: `5` ở `0` -> `1 == 1` có -> `1`.

In [7]:
def positions(nums):
    pos = {}
    for i, x in enumerate(nums):
        pos.setdefault(x, []).append(i)
    return pos

for nums in ([3, 3, 1, 2, 2, 1], [1, 2, 2, 1], [5]):
    pos = positions(nums)
    print(nums, {x: (p[-1] - p[0] + 1 == len(p)) for x, p in pos.items()})

[3, 3, 1, 2, 2, 1] {3: True, 1: False, 2: True}
[1, 2, 2, 1] {1: False, 2: True}
[5] {5: True}


## 5. Cách 1 — Brute Force / baseline: kiểm tra trực tiếp theo định nghĩa

Với mỗi giá trị `x` khác nhau: lấy `lo = nums.index(x)` và `hi = vị trí cuối`; rồi kiểm tra **mọi** phần tử trong đoạn `nums[lo..hi]` đều bằng `x`. Đúng nghĩa đen ("mọi lần xuất hiện tạo một khối"), nhưng mỗi giá trị quét lại đoạn `O(n)`: tổng `O(n * d)`.

"Nút thắt": quét lặp lại; có thể gom bằng một lượt nhờ vị trí đầu/cuối hoặc đếm khối.

In [8]:
class SolutionBrute:
    def countSpecialIntegers(self, nums):
        count = 0
        for x in set(nums):
            lo = nums.index(x)
            hi = len(nums) - 1 - nums[::-1].index(x)
            if all(nums[k] == x for k in range(lo, hi + 1)):
                count += 1
        return count

print(SolutionBrute().countSpecialIntegers([1, 2, 2, 1]), SolutionBrute().countSpecialIntegers([3, 3, 1, 2, 2, 1]))

1 2


**Độ phức tạp:** Time `O(n * d)` với `d` số giá trị khác nhau (`<= n`), tức `O(n^2)` xấu nhất; Space `O(d)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** không cần quét lại đoạn. Chỉ cần **ba con số** cho mỗi giá trị: **vị trí đầu**, **vị trí cuối**, **số lần**. Một giá trị đặc biệt iff `cuối - đầu + 1 == số lần`.

(Vì các vị trí của `x` nằm trong `[đầu, cuối]` và đôi một khác nhau, số vị trí `<= cuối - đầu + 1`, dấu bằng khi và chỉ khi **mọi** ô trong đoạn đều là `x`.)

**Cách nhìn thứ hai (đếm khối):** giá trị đặc biệt iff nó chỉ có **một khối**. Đếm khối bằng `groupby` hoặc đếm số vị trí `i` mà `nums[i] == x` và (`i == 0` hoặc `nums[i-1] != x`) — "số điểm bắt đầu khối".

**Quy tắc:** `special(x) <=> blocks(x) == 1`.

**Invariant (quét một lượt):** các giá trị trong `closed` đã kết thúc ít nhất một khối; gặp lại giá trị trong `closed` nghĩa là xuất hiện khối thứ hai -> không đặc biệt.

## 7. Các cách tiếp theo

### Cách 2 — Vị trí đầu, vị trí cuối và đếm

Một lượt xây `first`, `last`, `cnt`; rồi đếm các giá trị có `last - first + 1 == cnt`.

In [9]:
class SolutionFirstLast:
    def countSpecialIntegers(self, nums):
        first, last, cnt = {}, {}, {}
        for i, x in enumerate(nums):
            first.setdefault(x, i)
            last[x] = i
            cnt[x] = cnt.get(x, 0) + 1
        return sum(1 for x in cnt if last[x] - first[x] + 1 == cnt[x])

print(SolutionFirstLast().countSpecialIntegers([1, 2, 2, 1]), SolutionFirstLast().countSpecialIntegers([3, 3, 1, 2, 2, 1]))

1 2


**Độ phức tạp:** Time `O(n)`, Space `O(d)`.

### Cách 3 — Đếm số khối mỗi giá trị bằng `groupby`

Mỗi nhóm từ `groupby` là một khối; `blocks[v] += 1`; đáp án là số giá trị có đúng một khối.

In [10]:
from itertools import groupby

class SolutionGroupBy:
    def countSpecialIntegers(self, nums):
        blocks = {}
        for v, _ in groupby(nums):
            blocks[v] = blocks.get(v, 0) + 1
        return sum(1 for b in blocks.values() if b == 1)

print(SolutionGroupBy().countSpecialIntegers([1, 2, 2, 1]), SolutionGroupBy().countSpecialIntegers([3, 3, 1, 2, 2, 1]))

1 2


**Độ phức tạp:** Time `O(n)`, Space `O(d)`.

### Cách 4 — Quét một lượt với tập `closed` và `bad`

Không dùng từ điển vị trí: giữ `prev`, tập `closed` (giá trị đã kết thúc một khối), tập `bad` (giá trị xuất hiện lại sau khi đóng). Đáp án = số giá trị khác nhau trừ số giá trị trong `bad`.

In [11]:
class SolutionOnePass:
    def countSpecialIntegers(self, nums):
        closed, bad, seen = set(), set(), set()
        prev = None
        for x in nums:
            seen.add(x)
            if x != prev:                    # bắt đầu một khối mới
                if prev is not None:
                    closed.add(prev)          # khối của prev vừa kết thúc
                if x in closed:
                    bad.add(x)                # x đã có một khối trước đó -> khối thứ hai
            prev = x
        return len(seen) - len(bad)

print(SolutionOnePass().countSpecialIntegers([1, 2, 2, 1]), SolutionOnePass().countSpecialIntegers([3, 3, 1, 2, 2, 1]))

1 2


**Độ phức tạp:** Time `O(n)`, Space `O(d)` (ba tập).

## 8. TRACE từng bước

Trace Cách 4 trên `nums = [3, 3, 1, 2, 2, 1]`:

| `x` | `prev` | bắt đầu khối mới? | `closed` sau | `bad` sau |
|---|---|---|---|---|
| 3 | None | có (đầu) | `{}` | `{}` |
| 3 | 3 | không | `{}` | `{}` |
| 1 | 3 | có: đóng 3 | `{3}` | `{}` |
| 2 | 1 | có: đóng 1 | `{3, 1}` | `{}` |
| 2 | 2 | không | `{3, 1}` | `{}` |
| 1 | 2 | có: đóng 2; `1 in closed` | `{3, 1, 2}` | `{1}` |

`seen = {3, 1, 2}`, `bad = {1}` -> `3 - 1 = 2`.

In [12]:
def trace_one_pass(nums):
    closed, bad, seen = set(), set(), set()
    prev = None
    print("nums =", nums)
    for x in nums:
        seen.add(x)
        note = "tiếp tục khối"
        if x != prev:
            if prev is not None:
                closed.add(prev)
            if x in closed:
                bad.add(x)
                note = "KHỐI THỨ HAI của giá trị đã đóng"
            else:
                note = "bắt đầu khối mới"
        print(f"  x={x}: {note:<34} closed={sorted(closed)}, bad={sorted(bad)}")
        prev = x
    return len(seen) - len(bad)

assert trace_one_pass([3, 3, 1, 2, 2, 1]) == 2
assert trace_one_pass([1, 2, 2, 1]) == 1
assert trace_one_pass([5]) == 1

nums = [3, 3, 1, 2, 2, 1]
  x=3: bắt đầu khối mới                   closed=[], bad=[]
  x=3: tiếp tục khối                      closed=[], bad=[]
  x=1: bắt đầu khối mới                   closed=[3], bad=[]
  x=2: bắt đầu khối mới                   closed=[1, 3], bad=[]
  x=2: tiếp tục khối                      closed=[1, 3], bad=[]
  x=1: KHỐI THỨ HAI của giá trị đã đóng   closed=[1, 2, 3], bad=[1]
nums = [1, 2, 2, 1]
  x=1: bắt đầu khối mới                   closed=[], bad=[]
  x=2: bắt đầu khối mới                   closed=[1], bad=[]
  x=2: tiếp tục khối                      closed=[1], bad=[]
  x=1: KHỐI THỨ HAI của giá trị đã đóng   closed=[1, 2], bad=[1]
nums = [5]
  x=5: bắt đầu khối mới                   closed=[], bad=[]


In [13]:
def trace_first_last(nums):
    first, last, cnt = {}, {}, {}
    for i, x in enumerate(nums):
        first.setdefault(x, i); last[x] = i; cnt[x] = cnt.get(x, 0) + 1
    res = 0
    for x in cnt:
        ok = last[x] - first[x] + 1 == cnt[x]
        print(f"  x={x}: first={first[x]}, last={last[x]}, cnt={cnt[x]}, last-first+1={last[x] - first[x] + 1} -> {'đặc biệt' if ok else 'không'}")
        res += ok
    return res

assert trace_first_last([3, 3, 1, 2, 2, 1]) == 2

  x=3: first=0, last=1, cnt=2, last-first+1=2 -> đặc biệt
  x=1: first=2, last=5, cnt=2, last-first+1=4 -> không
  x=2: first=3, last=4, cnt=2, last-first+1=2 -> đặc biệt


## 9. Code nộp LeetCode

Dùng Cách 2 (vị trí đầu/cuối + đếm), đơn giản và dễ chứng minh.

In [14]:
from typing import List, Optional


class Solution:
    def countSpecialIntegers(self, nums: List[int]) -> int:
        first, last, cnt = {}, {}, {}
        for i, x in enumerate(nums):
            first.setdefault(x, i)
            last[x] = i
            cnt[x] = cnt.get(x, 0) + 1
        return sum(1 for x in cnt if last[x] - first[x] + 1 == cnt[x])


print(Solution().countSpecialIntegers([1, 2, 2, 1]), Solution().countSpecialIntegers([3, 3, 1, 2, 2, 1]))

1 2


## 10. Vì sao thuật toán đúng

Xét một giá trị `x` có `c` lần xuất hiện ở các vị trí `p_1 < p_2 < ... < p_c`, với `first = p_1`, `last = p_c`.

1. **Chặn trên.** Các vị trí đôi một khác nhau và đều trong `[first, last]`, đoạn này có `last - first + 1` ô, nên `c <= last - first + 1`.
2. **Dấu bằng.** `c = last - first + 1` khi và chỉ khi **mọi** ô của đoạn `[first, last]` là một vị trí của `x`, tức đoạn `nums[first..last]` toàn `x`.
3. **Khối liên tiếp.** "Mọi lần xuất hiện của `x` tạo một đoạn con liên tiếp" nghĩa là tập vị trí là một khoảng nguyên `[a, b]`. Khi đó `first = a`, `last = b`, `c = b - a + 1 = last - first + 1`. Ngược lại, nếu `c = last - first + 1` thì (2) cho thấy tập vị trí là cả `[first, last]`, một khoảng nguyên. Vậy điều kiện `last - first + 1 == c` **tương đương** với định nghĩa đặc biệt.
4. **Đếm đúng.** Ta duyệt qua các **giá trị khác nhau** (từng khóa của `cnt`), mỗi giá trị đúng một lần, nên kết quả là số giá trị đặc biệt (không phải số khối).
5. **Cách 3 và 4** đếm khối: giá trị đặc biệt iff có đúng một khối (tập vị trí là khoảng nguyên duy nhất); Cách 1 kiểm tra trực tiếp nên cho cùng kết quả.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums) <= 100` và `d <= 100` số giá trị khác nhau.

- Cách 1 (vét cạn): Time `O(n * d)`, Space `O(d)`.
- Cách 2 (nộp): Time `O(n)`, Space `O(d)`.
- Cách 3 (`groupby`): Time `O(n)`, Space `O(d)`.
- Cách 4 (quét một lượt): Time `O(n)`, Space `O(d)`.

`O(n)` là tối ưu vì phải đọc mọi phần tử; bộ nhớ `O(d)` là cần thiết để nhớ trạng thái từng giá trị (hoặc `O(1)` nếu giá trị bị chặn nhỏ như `<= 100`: mảng cố định kích thước 101).

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [15]:
import random

solutions = [SolutionBrute(), SolutionFirstLast(), SolutionGroupBy(), SolutionOnePass(), Solution()]

fixed = [
    ([1, 2, 2, 1], 1), ([3, 3, 1, 2, 2, 1], 2), ([5], 1), ([1, 1, 1], 1), ([1, 2, 3], 3),
    ([1, 2, 1], 1), ([1, 2, 1, 2], 0), ([1, 1, 2, 2, 1, 1], 1), ([4, 4, 5, 6, 6, 4], 2),
    ([7, 8, 7, 8, 7, 8], 0), ([1, 2, 2, 3, 3, 3, 2], 2), ([100, 1, 100], 1),
    ([1] + [2] * 98 + [1], 1),
]
for nums, expected in fixed:
    for sol in solutions:
        assert sol.countSpecialIntegers(list(nums)) == expected, (type(sol).__name__, nums)
print("test cố định OK:", len(fixed), "ca")

random.seed(4038)
for _ in range(1000):
    n = random.randint(1, 100)
    hi = random.choice([2, 3, 5, 10, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    expected = SolutionBrute().countSpecialIntegers(list(nums))
    for sol in solutions:
        assert sol.countSpecialIntegers(list(nums)) == expected, (type(sol).__name__, nums)
print("test ngẫu nhiên OK")

test cố định OK: 13 ca
test ngẫu nhiên OK


### Test thuộc tính (property test)

(1) Sắp xếp `nums` làm **mọi** giá trị trở nên đặc biệt (đáp án = số giá trị khác nhau). (2) Đảo ngược mảng không đổi kết quả. (3) Đáp án `<=` số giá trị khác nhau; với mảng một giá trị thì đáp án `1`. (4) Đổi tên giá trị theo song ánh không đổi kết quả.

In [16]:
random.seed(40380)
for _ in range(500):
    n = random.randint(1, 40)
    nums = [random.randint(1, 8) for _ in range(n)]
    r = Solution().countSpecialIntegers(nums)
    d = len(set(nums))
    assert r <= d
    assert Solution().countSpecialIntegers(sorted(nums)) == d
    assert Solution().countSpecialIntegers(nums[::-1]) == r
    assert Solution().countSpecialIntegers([101 - x for x in nums]) == r
assert Solution().countSpecialIntegers([9] * 50) == 1
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đếm **số khối** thay vì **số giá trị đặc biệt**

`[3,3,1,2,2,1]` có `4` khối nhưng chỉ `2` giá trị đặc biệt.

In [17]:
def wrong_count_blocks(nums):
    return sum(1 for _ in groupby(nums))                         # SAI: số khối

print(wrong_count_blocks([3, 3, 1, 2, 2, 1]), "!=", Solution().countSpecialIntegers([3, 3, 1, 2, 2, 1]))
assert wrong_count_blocks([3, 3, 1, 2, 2, 1]) == 4 and Solution().countSpecialIntegers([3, 3, 1, 2, 2, 1]) == 2

4 != 2


### Lỗi 2: nhầm với "xuất hiện **đúng một lần**"

Giá trị xuất hiện nhiều lần nhưng **liền nhau** vẫn đặc biệt (như `2,2`). Chỉ đếm giá trị có `count == 1` bỏ sót.

In [18]:
def wrong_only_once(nums):
    c = Counter(nums)
    return sum(1 for v in c.values() if v == 1)                  # SAI

print(wrong_only_once([1, 2, 2, 1]), "!=", Solution().countSpecialIntegers([1, 2, 2, 1]))
assert wrong_only_once([1, 2, 2, 1]) == 0 and Solution().countSpecialIntegers([1, 2, 2, 1]) == 1

0 != 1


### Lỗi 3: chỉ so với hàng xóm trái (`nums[i] != nums[i-1]`) để đếm giá trị

Đếm số vị trí bắt đầu khối chính là đếm khối (Lỗi 1). Phải **gom theo giá trị** rồi mới quyết định.

In [19]:
def wrong_neighbor_only(nums):
    return sum(1 for i, x in enumerate(nums) if i == 0 or nums[i - 1] != x)     # SAI: đếm khối

print(wrong_neighbor_only([1, 2, 1]), "!=", Solution().countSpecialIntegers([1, 2, 1]))
assert wrong_neighbor_only([1, 2, 1]) == 3 and Solution().countSpecialIntegers([1, 2, 1]) == 1

3 != 1


### Lỗi 4: dùng `set(nums)` rồi coi mọi giá trị đều đặc biệt

`len(set(nums))` bỏ qua vị trí. Đúng chỉ khi mảng đã sắp xếp (hoặc mọi giá trị liền nhau).

In [20]:
def wrong_set(nums):
    return len(set(nums))                                          # SAI: mất thông tin vị trí

print(wrong_set([1, 2, 1, 2]), "!=", Solution().countSpecialIntegers([1, 2, 1, 2]))
assert wrong_set([1, 2, 1, 2]) == 2 and Solution().countSpecialIntegers([1, 2, 1, 2]) == 0
assert wrong_set(sorted([1, 2, 1, 2])) == Solution().countSpecialIntegers(sorted([1, 2, 1, 2]))

2 != 0


### Lỗi 5: điều kiện `last - first == cnt` (thiếu `+ 1`)

Đoạn `[first, last]` có `last - first + 1` ô. Thiếu `+ 1` làm lệch một.

In [21]:
def wrong_off_by_one(nums):
    first, last, cnt = {}, {}, {}
    for i, x in enumerate(nums):
        first.setdefault(x, i); last[x] = i; cnt[x] = cnt.get(x, 0) + 1
    return sum(1 for x in cnt if last[x] - first[x] == cnt[x])      # SAI: thiếu + 1

print(wrong_off_by_one([1, 2, 2, 1]), "!=", Solution().countSpecialIntegers([1, 2, 2, 1]))
assert wrong_off_by_one([1, 2, 2, 1]) == 0 and Solution().countSpecialIntegers([1, 2, 2, 1]) == 1

0 != 1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mọi lần xuất hiện **liền nhau** / một khối" -> so `last - first + 1` với `count`, hoặc **đếm số khối** bằng `groupby`.
- Đếm theo **giá trị khác nhau**: lặp qua khóa của từ điển, không lặp qua khối.
- Mẫu "quét một lượt với `prev` và tập `closed`" dùng cho mọi bài "khối đã kết thúc không được xuất hiện lại" (ví dụ kiểm tra chuỗi dạng `aaabbbccc`).
- Giá trị đơn lẻ cũng là một khối (độ dài 1): coi chừng ca biên.
- Mảng đã sắp xếp: mọi giá trị đều đặc biệt (mảng đã nhóm theo giá trị).
- Ràng buộc nhỏ (`n <= 100`): có thể vét cạn để kiểm chứng (như `SolutionBrute`).

## 15. Biến thể của bài

1. **Check if Binary String Has at Most One Segment of Ones** (1784): đúng một khối `1`.
2. **Consecutive Characters** (1446), **Count Binary Substrings** (696): làm việc với khối liên tiếp.
3. **Remove All Adjacent Duplicates In String** (1047): xử lý các khối liền kề.
4. **Contains Duplicate** (217), **Longest Substring Without Repeating Characters** (3): làm việc với vị trí/số lần của giá trị.
5. Biến thể: đếm giá trị mà tối thiểu `2` khối (không đặc biệt); hoặc tổng độ dài các khối lớn nhất của mỗi giá trị.
6. Biến thể: với mảng **vòng** (phần tử cuối nối với đầu), `[1,2,1]` thì `1` là một khối vòng.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn theo định nghĩa | kiểm tra đoạn `[đầu, cuối]` toàn `x` | `O(n d)` | `O(d)` | Trọng tài, `n` nhỏ |
| 2. Đầu/cuối + đếm | `cuối - đầu + 1 == đếm` | `O(n)` | `O(d)` | **Nộp bài** |
| 3. `groupby` | số khối mỗi giá trị `== 1` | `O(n)` | `O(d)` | Nghĩ theo "khối" |
| 4. Quét một lượt | tập `closed` / `bad` | `O(n)` | `O(d)` | Khi cần dòng dữ liệu (streaming) |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().countSpecialIntegers([4, 4, 4, 5, 5, 4])` và `Solution().countSpecialIntegers([1, 2, 3, 2])` ra gì?

**Bài 2 (tính tay):** với `[2, 2, 3, 1, 3, 3]`, lập bảng `first`, `last`, `cnt` cho từng giá trị.

**Bài 3 (điền TODO):** hoàn thành `count_non_special(nums)`: số giá trị **không** đặc biệt.

**Bài 4 (tự viết lại):** viết `longest_special_block(nums)`: độ dài khối dài nhất trong các giá trị đặc biệt (0 nếu không có).

**Bài 5 (mini challenge):** viết `count_special_circular(nums)`: coi mảng **vòng** (cuối nối đầu); `[1,2,1]` thì `1` đặc biệt.

In [22]:
# Bài 1: đoán rồi chạy
print(Solution().countSpecialIntegers([4, 4, 4, 5, 5, 4]), Solution().countSpecialIntegers([1, 2, 3, 2]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_non_special_todo(nums):
    # TODO: số giá trị khác nhau trừ số giá trị đặc biệt
    return 0

1 2


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: nhìn từng giá trị xem có hai khối rời nhau không. Bài 2: viết các vị trí của `2`, `3`, `1`.

**Mức 2 (vừa):** Bài 3: `len(set(nums)) - Solution().countSpecialIntegers(nums)`. Bài 4: kèm độ dài `cnt[x]` cho giá trị đặc biệt, lấy `max`. Bài 5: nếu `nums[0] == nums[-1]` thì khối cuối và khối đầu của giá trị đó thuộc **cùng** một khối vòng; cần trừ một khối.

**Mức 3 (gần đáp án):** Bài 1: `1` (giá trị `5`; `4` có hai khối) và `2` (`1` và `3`; `2` có hai khối). Bài 2: `2` ở chỉ số `0, 1` (một khối), `3` ở `2, 4, 5` (hai khối), `1` ở `3` (một khối) -> đáp án `2`. Bài 3: `[2,2,3,1,3,3] -> 1`. Bài 5: `[1,2,1] -> 2`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().countSpecialIntegers([4, 4, 4, 5, 5, 4]) == 1 and Solution().countSpecialIntegers([1, 2, 3, 2]) == 2
nums2 = [2, 2, 3, 1, 3, 3]
assert positions(nums2) == {2: [0, 1], 3: [2, 4, 5], 1: [3]}
assert Solution().countSpecialIntegers(nums2) == 2                  # 2 và 1 đặc biệt; 3 có hai khối

# Bài 3
def count_non_special(nums):
    return len(set(nums)) - Solution().countSpecialIntegers(nums)

assert count_non_special(nums2) == 1 and count_non_special([1, 2, 1, 2]) == 2 and count_non_special([5]) == 0

# Bài 4
def longest_special_block(nums):
    first, last, cnt = {}, {}, {}
    for i, x in enumerate(nums):
        first.setdefault(x, i); last[x] = i; cnt[x] = cnt.get(x, 0) + 1
    lens = [cnt[x] for x in cnt if last[x] - first[x] + 1 == cnt[x]]
    return max(lens, default=0)

assert longest_special_block([3, 3, 1, 2, 2, 1]) == 2 and longest_special_block([1, 2, 1, 2]) == 0
assert longest_special_block([1] * 5 + [2]) == 5

# Bài 5
def count_special_circular(nums):
    if len(set(nums)) == 1:
        return 1
    k = 0
    while nums[k - 1] == nums[k]:        # xoay để bắt đầu tại ranh giới một khối
        k += 1
    rot = nums[k:] + nums[:k]
    return Solution().countSpecialIntegers(rot)

assert count_special_circular([1, 2, 1]) == 2 and count_special_circular([1, 2, 1, 2]) == 0
assert count_special_circular([1, 1]) == 1 and count_special_circular([3, 1, 3, 3]) == 2
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def countSpecialIntegers(self, nums: List[int]) -> int:
        first, last, cnt = {}, {}, {}
        for i, x in enumerate(nums):
            first.setdefault(x, i)          # vị trí đầu tiên
            last[x] = i                     # vị trí cuối cùng (ghi đè)
            cnt[x] = cnt.get(x, 0) + 1      # số lần
        return sum(1 for x in cnt if last[x] - first[x] + 1 == cnt[x])   # các vị trí lấp kín [đầu, cuối]
```

Khung chung "mọi lần xuất hiện liền nhau": `cuối - đầu + 1 == đếm` hoặc `số khối == 1`.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Đếm số **giá trị khác nhau** đặc biệt (không phải số khối); công thức có `+ 1`.
- [ ] Đã thử mảng một phần tử, mảng toàn một giá trị, `[1,2,1]`, `[1,2,1,2]` và mảng đã sắp xếp.